In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats
import boto3
import os
from tqdm import tqdm
import s3fs
import joblib
import warnings
import optuna
from optuna.samplers import TPESampler
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)


## Helper Classes

In [ ]:
class SarimaModel:
    """SARIMA model development and evaluation."""
    
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.fs = s3fs.S3FileSystem()
        self.s3_client = boto3.client('s3')
        self.df_train = None
        self.df_test = None
        self.model = None
        self.model_results = None
        self.forecast = None
        self.tuple_best_order = None
        self.tuple_best_seasonal_order = None
        self.study = None
    
    def import_data(self):
        """Load train/test splits from S3."""
        str_train_uri = f's3://{self.str_bucket}/02_preprocessing/train_data.csv'
        str_test_uri = f's3://{self.str_bucket}/02_preprocessing/test_data.csv'
        
        self.df_train = pd.read_csv(str_train_uri)
        self.df_test = pd.read_csv(str_test_uri)
        
        self.df_train['date'] = pd.to_datetime(self.df_train['date'])
        self.df_test['date'] = pd.to_datetime(self.df_test['date'])
        
        print(f'Loaded train: {len(self.df_train)} samples')
        print(f'Loaded test: {len(self.df_test)} samples')
        
        return self.df_train, self.df_test
    
    def _objective(self, trial):
        """Optuna objective function to minimize AIC."""
        # Suggest parameters
        int_p = trial.suggest_int('p', 0, 2)
        int_d = trial.suggest_int('d', 0, 1)
        int_q = trial.suggest_int('q', 0, 2)
        int_P = trial.suggest_int('P', 0, 1)
        int_D = trial.suggest_int('D', 0, 1)
        int_Q = trial.suggest_int('Q', 0, 1)
        int_s = 12  # Fixed seasonal period
        
        try:
            model = SARIMAX(
                self.df_train['attrition_rate'],
                order=(int_p, int_d, int_q),
                seasonal_order=(int_P, int_D, int_Q, int_s),
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            result = model.fit(disp=False)
            return result.aic
        except Exception as e:
            # If fitting fails, return large penalty
            return 1e10
    
    def optuna_search_order(self, int_n_trials=20):
        """Use Optuna to search parameter space."""
        print('
Performing Optuna hyperparameter search...')
        
        # Create study with TPESampler
        sampler = TPESampler(seed=42)
        self.study = optuna.create_study(
            sampler=sampler,
            direction='minimize'
        )
        
        # Optimize
        self.study.optimize(self._objective, n_trials=int_n_trials, show_progress_bar=True)
        
        # Get best trial
        best_trial = self.study.best_trial
        int_p = best_trial.params['p']
        int_d = best_trial.params['d']
        int_q = best_trial.params['q']
        int_P = best_trial.params['P']
        int_D = best_trial.params['D']
        int_Q = best_trial.params['Q']
        int_s = 12
        
        self.tuple_best_order = (int_p, int_d, int_q)
        self.tuple_best_seasonal_order = (int_P, int_D, int_Q, int_s)
        
        print(f'
Optuna Search Complete!')
        print(f'Best order: {self.tuple_best_order}')
        print(f'Best seasonal order: {self.tuple_best_seasonal_order}')
        print(f'Best AIC: {best_trial.value:.4f}')
        
        # Display top trials
        df_trials = self.study.trials_dataframe()
        df_trials_sorted = df_trials.sort_values('value').head(5)
        print(f'
Top 5 trials:')
        print(df_trials_sorted[['number', 'value', 'params_p', 'params_d', 'params_q', 'params_P', 'params_D', 'params_Q']].to_string(index=False))
        
        return df_trials
    
    def fit_model(self):
        """Fit SARIMA with best parameters."""
        print(f'
Fitting SARIMA{self.tuple_best_order}x{self.tuple_best_seasonal_order}...')
        
        self.model = SARIMAX(self.df_train['attrition_rate'],
                            order=self.tuple_best_order,
                            seasonal_order=self.tuple_best_seasonal_order,
                            enforce_stationarity=False,
                            enforce_invertibility=False)
        
        self.model_results = self.model.fit(disp=False)
        
        print(f'
Model Summary:')
        print(self.model_results.summary())
        
        return self.model_results
    
    def diagnose(self):
        """Residual diagnostics."""
        residuals = self.model_results.resid
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # Residuals over time
        axes[0, 0].plot(residuals, linewidth=1)
        axes[0, 0].set_title('Residuals Over Time', fontsize=12, fontweight='bold')
        axes[0, 0].set_ylabel('Residual')
        axes[0, 0].grid(True, alpha=0.3)
        axes[0, 0].axhline(y=0, color='r', linestyle='--', alpha=0.5)
        
        # Histogram with KDE
        axes[0, 1].hist(residuals, bins=20, density=True, alpha=0.6, edgecolor='black')
        residuals.plot(kind='kde', ax=axes[0, 1], secondary_y=False, linewidth=2, color='red')
        axes[0, 1].set_title('Distribution of Residuals', fontsize=12, fontweight='bold')
        axes[0, 1].set_xlabel('Residual')
        axes[0, 1].grid(True, alpha=0.3)
        
        # Q-Q plot
        stats.probplot(residuals, dist='norm', plot=axes[1, 0])
        axes[1, 0].set_title('Q-Q Plot', fontsize=12, fontweight='bold')
        axes[1, 0].grid(True, alpha=0.3)
        
        # ACF of residuals
        plot_acf(residuals, lags=40, ax=axes[1, 1])
        axes[1, 1].set_title('Autocorrelation of Residuals', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        str_path = f'{self.str_dirname_output}/11_sarima_diagnostics.png'
        plt.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'
Saved diagnostics to {str_path}')
        plt.show()
    
    def forecast(self, int_steps=12):
        """Generate forecast with confidence intervals."""
        self.forecast = self.model_results.get_forecast(steps=int_steps)
        df_forecast = self.forecast.conf_int(alpha=0.05)
        df_forecast['forecast'] = self.forecast.predicted_mean
        df_forecast.columns = ['ci_lower', 'ci_upper', 'forecast']
        
        print(f'
Generated {int_steps}-step forecast')
        print(df_forecast.head(10))
        
        return df_forecast
    
    def plot_forecast(self, df_forecast):
        """Plot actual vs forecast with CI bands."""
        fig, ax = plt.subplots(figsize=(14, 7))
        
        # Historical data
        ax.plot(self.df_train.index, self.df_train['attrition_rate'], linewidth=2, 
               label='Training Data', marker='o', markersize=4)
        ax.plot(self.df_test.index + len(self.df_train), self.df_test['attrition_rate'], 
               linewidth=2, label='Test Data', marker='s', markersize=4, color='#ff7f0e')
        
        # Forecast
        int_forecast_idx = np.arange(len(self.df_train), len(self.df_train) + len(df_forecast))
        ax.plot(int_forecast_idx, df_forecast['forecast'], linewidth=2.5, 
               label='SARIMA Forecast', marker='^', markersize=5, color='#d62728')
        
        # Confidence intervals
        ax.fill_between(int_forecast_idx, df_forecast['ci_lower'], df_forecast['ci_upper'],
                        alpha=0.3, color='#d62728', label='95% Confidence Interval')
        
        ax.set_xlabel('Time Index', fontsize=11)
        ax.set_ylabel('Attrition Rate', fontsize=11)
        ax.set_title('SARIMA Forecast vs Actual', fontsize=13, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        
        str_path = f'{self.str_dirname_output}/12_sarima_forecast.png'
        plt.savefig(str_path, bbox_inches='tight', dpi=150)
        print(f'Saved forecast plot to {str_path}')
        plt.show()
    
    def evaluate(self, df_forecast):
        """Calculate RMSE, MAE, MAPE on test set."""
        arr_actual = self.df_test['attrition_rate'].values
        arr_pred = df_forecast['forecast'].values[:len(arr_actual)]
        
        flt_rmse = np.sqrt(np.mean((arr_actual - arr_pred) ** 2))
        flt_mae = np.mean(np.abs(arr_actual - arr_pred))
        flt_mape = np.mean(np.abs((arr_actual - arr_pred) / arr_actual)) * 100
        
        dict_metrics = {
            'RMSE': flt_rmse,
            'MAE': flt_mae,
            'MAPE': flt_mape
        }
        
        print(f'
Test Set Evaluation:')
        print(f'  RMSE: {flt_rmse:.6f}')
        print(f'  MAE: {flt_mae:.6f}')
        print(f'  MAPE: {flt_mape:.2f}%')
        
        return dict_metrics
    
    def save_model(self):
        """Serialize model to disk and S3."""
        str_model_path = f'{self.str_dirname_output}/sarima_model.pkl'
        joblib.dump(self.model_results, str_model_path)
        
        try:
            self.s3_client.upload_file(
                str_model_path,
                self.str_bucket,
                '03_sarima/sarima_model.pkl'
            )
            print(f'Saved model to {str_model_path} and S3')
        except Exception as e:
            print(f'Error uploading to S3: {e}')


## Constants

In [ ]:
str_bucket = 'time-series-forecasting-demo-repo'
str_task = 'employee_attrition_forecasting'
str_dirname_output = './output'

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

print(f'Output directory: {str_dirname_output}')

## Load Data

In [ ]:
sarima = SarimaModel(str_bucket, str_dirname_output)
df_train, df_test = sarima.import_data()

## Optuna Hyperparameter Search


In [ ]:
df_optuna_results = sarima.optuna_search_order(int_n_trials=20)


## Fit Best Model

In [ ]:
result = sarima.fit_model()

## Diagnostic Checks

In [ ]:
sarima.diagnose()

## Generate Forecast

## Plot Forecast

In [ ]:
sarima.plot_forecast(df_forecast)

## Evaluate Model

## Save Model